# Aprendizado de Máquina Distribuído com SparkML

Neste módulo, vamos entender como escalar o treinamento de modelos para cenários de Big Data. Quando o volume de dados supera a capacidade de memória RAM de uma única máquina (o nó "Driver"), o uso de bibliotecas tradicionais como o Scikit-Learn torna-se inviável, resultando em erros de falta de memória ("Out of Memory - OOM").

### Scikit-Learn vs. SparkML: Quando usar?

* **Scikit-Learn:** Ideal para datasets pequenos ou médios que cabem inteiramente na memória RAM de uma única máquina. Possui uma variedade gigantesca de algoritmos e flexibilidade de pré-processamento.
* **SparkML:** Projetado especificamente para Big Data. O processamento e o treinamento dos algoritmos são distribuídos nativamente entre todos os nós trabalhadores ("Workers") do cluster.


In [0]:
%%capture
%pip install databricks
%pip install hyperopt
%pip install optuna
dbutils.library.restartPython()

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Definição do caminho da tabela Iris
tabela_iris = "workspace.default.iris_dataset"

df_iris = spark.table(tabela_iris)

# Divisão de treino e teste (80% / 20%) distribuída
df_treino, df_teste = df_iris.randomSplit([0.8, 0.2], seed=42)

# O SparkML exige que o target seja numérico e as features estejam agrupadas em um único vetor
string_indexer = StringIndexer(inputCol="species", outputCol="label")

colunas_features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
assembler = VectorAssembler(inputCols=colunas_features, outputCol="features")

# Configurando o Classificador (Random Forest do SparkML)
rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)

# Criando e executando o Pipeline estruturado
pipeline = Pipeline(stages=[string_indexer, assembler, rf])

print("Treinando o modelo de baseline com SparkML...")
modelo_pipeline = pipeline.fit(df_treino)

# Avaliação das predições no conjunto de teste
predicoes = modelo_pipeline.transform(df_teste)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
acuracia_baseline = evaluator.evaluate(predicoes)

print(f"Acurácia do Baseline SparkML: {acuracia_baseline:.4f}")

Nesta célula, convertemos a tabela do Unity Catalog para um DataFrame do Pandas tradicional e separamos as variáveis para o treinamento.

Como o nosso dataset é relativamente pequeno, não precisamos utilizar o SparkML

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Definição do caminho da tabela Iris
tabela_iris = "workspace.default.iris_dataset"
df_iris = spark.table(tabela_iris).toPandas()

X = df_iris[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df_iris["species"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dados prontos! Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")

### Hyperopt: Tuning de Hiperparâmetros
Para encontrar os melhores parâmetros do modelo sem testar combinação por combinação manualmente (como no "GridSearch"), utilizaremos o **Hyperopt**.

A função `fmin()` é o coração do Hyperopt. É ela quem executa a otimização bayesiana, cruzando o espaço de busca com a sua função objetivo para encontrar a combinação de parâmetros que minimiza a perda (*loss*).

### Parâmetros Detalhados

* **`fn`**: A função Python que treina o modelo, avalia a métrica e retorna a perda numérica. O Hyperopt sempre tentará **minimizar** o valor retornado por essa função.
* **`space`**: O dicionário contendo as variáveis e os intervalos de busca que você definiu utilizando as funções `hp.` (ex: `hp.quniform`).
* **`algo`**: O algoritmo de busca. O `tpe.suggest` (Tree-structured Parzen Estimator) é o padrão bayesiano que aprende com os testes anteriores para sugerir os próximos melhores parâmetros, em vez de chutar aleatoriamente.
* **`max_evals`**: O número máximo de tentativas (iterações) que o algoritmo vai rodar antes de parar. No exemplo, ele testará exatamente 8 combinações.
* **`trials`**: O objeto que armazena o histórico, os metadados e os resultados de cada uma das tentativas executadas. Como estamos rodando localmente em Scikit-Learn (sem Spark), usamos a classe `Trials()` padrão.

```

```

## MLflow: mlflow.start_run

O `mlflow.start_run` cria um contexto de execução (**"Run"**) para gravar parâmetros, métricas e artefatos de um modelo. Usar com o bloco `with` garante o fechamento automático do log.

### Principais Parâmetros

* **`run_name`**: Nome visível da tentativa na interface do Databricks.
* **`nested=True`**: Cria uma execução "filha" dentro de uma "pai". Essencial para organizar loops de hiperparâmetros (Optuna/Hyperopt), agrupando as tentativas de forma colapsável no painel.



In [0]:
import mlflow
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Ativa o autolog nativo do Scikit-Learn (sem salvar o modelo bruto em cada trial)
mlflow.sklearn.autolog(log_models=False)

# Espaço de busca para o Random Forest do Sklearn
espaco_busca = {
    'n_estimators': hp.quniform('n_estimators', 10, 80, 1),
    'max_depth': hp.quniform('max_depth', 3, 10, 1)
}

# Função objetivo usando Scikit-Learn puro
def funcao_objetivo(params):
    n_estimators = int(params['n_estimators'])
    max_depth = int(params['max_depth'])
    
    # Criamos uma run filha para organizar o dashboard do MLflow
    with mlflow.start_run(run_name="Hyperopt_Sklearn_Trial", nested=True):
        
        # O autolog já grava n_estimators e max_depth automaticamente ao chamar o .fit()
        clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        clf.fit(X_train, y_train)
        
        # Avaliação
        predicoes = clf.predict(X_test)
        acuracia = accuracy_score(y_test, predicoes)
        
        # Forçamos o log da métrica final de teste de forma explícita
        mlflow.log_metric("test_accuracy", acuracia)
        
        perda = -acuracia
        return {'loss': perda, 'status': STATUS_OK}

# Uso do Trials() comum do Hyperopt
trials_comum = Trials()

print("Iniciando otimização com Hyperopt local...")
with mlflow.start_run(run_name="Hyperopt_Sklearn_Iris_Parent"):
    melhores_parametros = fmin(
        fn=funcao_objetivo,
        space=espaco_busca,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials_comum
    )

print(f"\nMelhores hiperparâmetros encontrados pelo Hyperopt: {melhores_parametros}")

## Otimização com Optuna

O **Optuna** baseia-se no conceito de "Next-Generation Optimization Framework". A sua principal vantagem sobre o Hyperopt é a **sintaxe imperativa**. Em vez de definir um espaço de busca estático no início, o espaço de busca do Optuna é definido dinamicamente dentro da própria função objetivo através de métodos simples como `trial.suggest_int` ou `trial.suggest_float`.

O MLflow possui integração nativa com o Optuna via `mlflow.optuna.autolog()`, gravando cada tentativa ("Trial") automaticamente no Unity Catalog.

In [0]:
import optuna
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def objetivo_optuna(trial):
    # Definição dinâmica do espaço de busca
    n_estimators = trial.suggest_int('n_estimators', 10, 80)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    
    with mlflow.start_run(run_name=f"Optuna_Sklearn_Trial_{trial.number}", nested=True):
        
        clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        clf.fit(X_train, y_train)
        
        predicoes = clf.predict(X_test)
        acuracia = accuracy_score(y_test, predicoes)
        
        mlflow.log_metric("test_accuracy", acuracia)
        
        return acuracia

print("Iniciando otimização com Optuna local...")
estudo = optuna.create_study(direction="maximize")

with mlflow.start_run(run_name="Optuna_Sklearn_Iris_Parent"):
    estudo.optimize(objetivo_optuna, n_trials=8)

print(f"\nMelhores parâmetros encontrados pelo Optuna: {estudo.best_params}")